# Similaridade tese × artigo

- O autor de cada artigo é procurado na tabela **completa** de teses (`data/materiais.csv`, todos os anos), mantendo apenas dissertações e teses.
- O texto do artigo é **título + resumo** (`resumo_scopus`); `usa_resumo` indica se o resumo estava disponível. `cosine_similarity_titulo` mantém a versão só com título para comparação.
- A lógica está em `src/similaridade_teses.py`.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from sentence_transformers import SentenceTransformer
from src.similaridade_teses import MODELO, autores_sem_tese, calcular_similaridade, carregar_artigos, carregar_teses

model = SentenceTransformer(MODELO)
theses = carregar_teses()

In [ ]:
PERIODOS = {
    "2025": (
        RAIZ / "RPVM_2026_10_05/outputs/alunos_artigos_scival_completo.csv",
        RAIZ / "data/thesis_articles_similarity.csv",
    ),
    "2021-2024": (
        RAIZ / "resultados_2021_2024/alunos_artigos_2021_2024_final.csv",
        RAIZ / "resultados_2021_2024/thesis_articles_similarity_2021_2024.csv",
    ),
}

resultados = {}
for periodo, (entrada, saida) in PERIODOS.items():
    articles = carregar_artigos(entrada, theses)
    similarity = calcular_similaridade(articles, theses, model)
    similarity.to_csv(saida, index=False)
    resultados[periodo] = similarity
    print(f"{periodo}: {len(similarity)} pares; {articles['usa_resumo'].sum()}/{len(articles)} artigos com resumo; "
          f"sem tese: {autores_sem_tese(articles)}")

In [ ]:
resultados["2021-2024"]

In [ ]:
summary = (
    resultados["2021-2024"].groupby("autor", as_index=False)
    .agg(n_articles=("doi", "nunique"), cosine_max=("cosine_similarity", "max"), cosine_mean=("cosine_similarity", "mean"))
    .sort_values("cosine_max", ascending=False)
    .reset_index(drop=True)
)
summary